## Computer Vision on Public Lands Webcams

#### Date: 11/12/2025
#### Author: Nineveh O'Connell

Goal: The goal of this notebook is to apply the YOLO computer vision tools to footage from the Acadia Sand Beach entrance station on Tuesday, August 26 to determine processing time at the entrance station.

In [1]:
#import libraries
import re
from datetime import datetime, timezone
import pandas as pd
import numpy as np

import time
from pathlib import Path
import os

import cv2
import yt_dlp
from ultralytics import YOLO
from collections import defaultdict
import supervision as sv
from bs4 import BeautifulSoup
import requests
from IPython.display import display, Image
from PIL import Image as Img
from PIL import ImageTk
from urllib.parse import urljoin


## YOLO modeling of video

In [3]:
def list_files_pathlib(directory_path_str):
    """
    Lists all files in the specified directory using pathlib.
    """
    directory_path = Path(directory_path_str)
    files = [str(p.resolve()) for p in directory_path.iterdir() if p.is_file()]
    return files

# Example usage (for current directory):
p_dir_base = "C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/4- Data Collection/Video Data/8.26.25_Park_Loop_1450_Dashcam/VIDEO_F/"
all_files_pathlib = list_files_pathlib(p_dir_base)



In [4]:
# Load the YOLO model
model = YOLO('yolo11l.pt')

class_list = model.names 

In [5]:

def get_video_properties_cv2(filename):
    video = cv2.VideoCapture(filename)
    
    if not video.isOpened():
        print(f"Error opening video file: {filename}")
        return None

    # Get dimensions
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Get length
    fps = video.get(cv2.CAP_PROP_FPS)
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps > 0:
        duration_seconds = frame_count / fps
    else:
        duration_seconds = 0

    video.release()

    return {
        "width": width,
        "height": height,
        "duration_seconds": duration_seconds
    }

# Example Usage
properties = get_video_properties_cv2(all_files_pathlib[0])
if properties:
    print(f"Dimensions: {properties['width']}x{properties['height']}")
    print(f"Length (seconds): {properties['duration_seconds']:.2f}s")


Dimensions: 2560x1440
Length (seconds): 60.04s


In [7]:
# 6) extract numeric confidence from a string like "label (0.82)" into confidence_numeric
#    regex captures the number inside parentheses (first occurrence)
# this will define that function
def extract_confidence(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\(([^)]+)\)", str(s))
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

In [ ]:
for video_path in all_files_pathlib:

    video_base_name = os.path.basename(video_path)
    out_csv_path = f"C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/7- Video Analysis/ParkLoopRd/cv_output{video_base_name}.csv"

    # if video has already been analyzed, break and move to next file in path
    if out_csv_path.is_file():
        break

    # if manual override, break and move to next file in path
    if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Open video
    cap = cv2.VideoCapture(video_path)
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    n_frames_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    curr_frame_num = 0

    # name window for viewer
    window_name = "Fullscreen Video"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    # to save results
    resultsList = []

    # the approach of using a while looks and checking for success isn't working
    # instead let's use a different while condition
    # this was the condition before: while cap.isOpened()

    while curr_frame_num < n_frames_total:
        ret, frame = cap.read()

        # Get current frame number
        frame_num = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        curr_frame_num = frame_num
        
        # If detections occur, do operations for every tenth frame 
        if frame_num % 10 == 0:

            if not ret:
                print("this is the for-loop internal call of")
                print("Video completed or error reading frame.")
                break

            # Process frame for detections
            results = model.track(frame, classes = [0,1,2,3,5,7,11], persist = True)

            if len(results) > 0:

                timestamp = frame_num / frame_rate
                cv2.putText(frame, f'Timestamp: {timestamp:.2f}s', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                            1, (255, 255, 255), 2, cv2.LINE_AA)
            
                # Here you can save the frame or timestamp if needed
                for r in results:

                    if r.boxes.id is not None:
                        boxes = r.boxes.xyxy.cpu()  # Boxes object for bbox outputs
                        print(boxes)
                        track_ids = r.boxes.id.int().cpu().tolist()
                        class_indices = r.boxes.cls.int().cpu().tolist()
                        confidences = r.boxes.conf.cpu()

                        # Loop through each detected object
                        for box, track_id, class_idx, conf in zip(boxes, track_ids, class_indices, confidences):
                            x1, y1, x2, y2 = map(int, box)
                            cx = (x1 + x2) // 2  # Calculate the center point
                            cy = (y1 + y2) // 2            

                            class_name = class_list[class_idx]

                            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                            
                            cv2.putText(frame, f"ID: {track_id} {class_name}", (x1, y1 - 10),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)
                            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) 

                            dfKeyFeatures = pd.DataFrame({'id' : [track_id], 
                                                        'class' : [class_name], 
                                                        'confidence' : [conf], 
                                                        'cx' : [cx], 
                                                        'cy' : [cy], 
                                                        'bottomLeftx' : [x1],
                                                        'bottomLefty' : [y1],
                                                        'upperRightx' : [x2],
                                                        'upperRighty' : [y2],
                                                        'timestamp' : [timestamp]})
                            resultsList.append(dfKeyFeatures)

                # Display the frame
                cv2.rectangle(frame, (1550, 950), (1950, 1250), (100, 80, 250), 2)

                cv2.imshow(window_name, frame)
            
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # compile results
    if(len(resultsList) > 1 ):
        combined_df = pd.concat(resultsList)
        combined_df["confidence_numeric"] = combined_df["confidence"].apply(extract_confidence)
        # export results to csv
        combined_df.to_csv(out_csv_path, index=False) 



0: 384x640 1 car, 627.2ms
Speed: 3.4ms preprocess, 627.2ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 562.2ms
Speed: 5.1ms preprocess, 562.2ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 614.7ms
Speed: 3.5ms preprocess, 614.7ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 611.5ms
Speed: 4.9ms preprocess, 611.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 592.9ms
Speed: 3.6ms preprocess, 592.9ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 591.9ms
Speed: 4.3ms preprocess, 591.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 608.5ms
Speed: 5.0ms preprocess, 608.5ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 560.6ms
Speed: 4.4ms preprocess, 560.6ms inference, 2.2ms postprocess per image at sha

ValueError: No objects to concatenate

In [ ]:
# compile results
combined_df = pd.concat(resultsList)

# 6) extract numeric confidence from a string like "label (0.82)" into confidence_numeric
#    regex captures the number inside parentheses (first occurrence)
def extract_confidence(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\(([^)]+)\)", str(s))
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

combined_df["confidence_numeric"] = combined_df["confidence"].apply(extract_confidence)

# export results to csv
video_base_name = os.path.basename(video_path)
combined_df.to_csv(f"C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/7- Video Analysis/ParkLoopRd/cv_output{video_base_name}.csv", index=False) 

## Defining Lanes

In [73]:
lane_boundaries = pd.DataFrame({
    "spot_id": np.arange(1, 5),
    "minx": [660, 1020, 1180, 1300],
    "maxx": [660, 920, 1150, 1300],
    "miny": [630, 560, 530, 450],
    "maxy" : [800, 650, 600, 400]
}).sort_values("minx").reset_index(drop=True)


Assign to parking spots, calling the parking spot -1 in the case of being in the roadway

In [74]:
combined_df['lane_id_minx'] = pd.cut(combined_df['cx'], bins = lane_boundaries['minx'], labels = np.arange(1,4), right = True)
combined_df['lane_id_maxx'] = pd.cut(combined_df['cx'], bins = lane_boundaries['maxx'], labels = np.arange(1,4), right = True)
#combined_df['lane_id_miny'] = pd.cut(combined_df['cy'], bins = lane_boundaries['miny'], labels = np.arange(1,4)[::-1], right = True)
#combined_df['lane_id_maxy'] = pd.cut(combined_df['cy'], bins = lane_boundaries['maxx'], labels = np.arange(1,4)[::-1], right = True)


Make confidence into a numeric variable and only keep instances with confidence over 0.35. From spot checking, instances with lower confidence are not really vehicles.

In [75]:
# 6) extract numeric confidence from a string like "label (0.82)" into confidence_numeric
#    regex captures the number inside parentheses (first occurrence)
def extract_confidence(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\(([^)]+)\)", str(s))
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

combined_df["confidence_numeric"] = combined_df["confidence"].apply(extract_confidence)

# # keep only rows where confidence is 0.35 or greater
# willow_creek_vehicles = combined_df[combined_df["confidence_numeric"] > 0.35]
# # Clean up column
# willow_creek_vehicles = willow_creek_vehicles.drop(columns=["confidence"])
# # make timestamp actual date time object, and turn id into a string
# willow_creek_vehicles['timestamp_dt'] = pd.to_datetime(willow_creek_vehicles['timestamp'], format='%Y-%m-%d %H-%M-%S')



In [ ]:
# export results to csv
combined_df.to_csv(f"C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/7- Video Analysis/ParkLoopRd/cv_output{video_file_name}.csv", index=False) 